In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
from jppype import vscode_theme
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset

vscode_theme()
# mp.set_start_method("spawn", force=True)

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [4]:
DATASETS_ROOT = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/")
DATASETS_PATH = {
    dataset: DATASETS_ROOT / folder
    for dataset, folder in {
        "GAVE-train": "GAVE-train",
        "MAPLES-DR": "MAPLES-DR",
        "FundusAV": "Fundus-AV",
        "HRF": "HRF",
        "LES-AV": "LES-AV",
        "INSPIRE": "INSPIRE",
        "DRIVE_train": "AV_DRIVE/training",
        "DRIVE_test": "AV_DRIVE/test",
    }.items()
}

RAW = [path / "1-images" for path in DATASETS_PATH.values()]
TOPO = [path / "3-topo" for path in DATASETS_PATH.values()]
AV = [
    {
        "fvt": path / "2-av-pred_FVT",
        "automorph": path / "2-av-pred_Automorph",
        "gt": path / "2-av",
        "vascx": path / "2-av-pred_VascX",
    }
    for path in DATASETS_PATH.values()
]

dataset = BranchDigraphDataset.load_from_dirs(
    RAW,
    TOPO,
    AV,
    dataset_name=list(DATASETS_PATH.keys()),
    resize_to=1024,
    output_dir="tmp/ALL_DATA_V2",
    n_workers=0,
    mask_optic_disc=True,
)

Found 373 branch digraphs...


Processing...
Done!


In [ ]:
dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")


## Graph Augment


In [ ]:
from fundus_vessels_toolkit.models.topology.data_augmentation import HSVJitterCfg

ID = 0
m, digraph, _ = dataset.jppype_show(ID)
m

In [5]:
dataset.get(20, augment=True, version="fvt")
%timeit dataset.get(20, augment=True, version="fvt")

251 ms ± 3.15 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
subset = dataset.split(list(range(20))).preload()
%timeit [subset.get(i, augment=True, version="fvt") for i in range(3)]

Output()

481 ms ± 14.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
from fundus_vessels_toolkit.utils.profiling import Profiler, watch

with Profiler():
    for i in range(len(subset)):
        subset.get(i, augment=True, version="fvt")

3


]8;id=129021;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:409\BranchDigraphData.from_graph]8;;\               2.7s                             (runs=20, avg= 135.9ms)
├── ]8;id=348681;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:410\parse cfg]8;;\                             ↳0.2‰  586.0µs                    (runs=20, avg=  29.3µs)
├── ]8;id=65972;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:413\graph preprocessing]8;;\                   ↳0.9‰    2.3ms                    (runs=20, avg= 115.6µs)
├── ]8;id=73670;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:418\graph deterioration]8;;\                   ↳ 11%  294.1ms                    (runs=20, avg=  14.7ms)
├── ]8;id=914955;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:421\VBranchDigraph.from_graph]8;;\             ↳ 29%  775.3ms                    (runs=20, avg=  38.8ms)
├── ]8;id=859259;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:424\compute_p_from_gt]8;;\                     ↳ 13%  348.1ms                    (runs=20, avg=  17.4ms)
├── ]8;id=214392;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:427\read fundus and preprocess]8;;\            ↳0.1%    3.0ms                    (runs=20, avg= 150.4µs)
├── ]8;id=256078;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:452\Color Augmentation]8;;\                    ↳ 18%  495.9ms                    (runs=20, avg=  24.8ms)
│   ├── ]8;id=405279;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:454\compute roi mask]8;;\                  1.62%    ↳8.9%   44.0ms           (runs=20, avg=   2.2ms)
│   ├── ]8;id=448208;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:456\mask roi]8;;\                          2.49%    ↳ 14%   67.6ms           (runs=20, avg=   3.4ms)
│   └── ]8;id=511078;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:458\hsv jitter]8;;\                          14%    ↳ 77%  384.1ms           (runs=20, avg=  19.2ms)
│       ├── ]8;id=10380;file://<string>:6\RGB to HSV]8;;\                    0.41%    2.26%    ↳2.9%   11.2ms  (runs=20, avg= 560.6µs)
│       ├── ]8;id=443891;file://<string>:8\hue jitter]8;;\                    0.42%    2.28%    ↳2.9%   11.3ms  (runs=20, avg= 566.0µs)
│       ├── ]8;id=174508;file://<string>:10\saturation jitter]8;;\             1.24%    6.80%    ↳8.8%   33.7ms  (runs=20, avg=   1.7ms)
│       ├── ]8;id=466321;file://<string>:14\value jitter]8;;\                  1.13%    6.19%    ↳8.0%   30.7ms  (runs=20, avg=   1.5ms)
│       ├── ]8;id=922535;file://<string>:18\HSV to RGB]8;;\                    0.26%    1.45%    ↳1.9%    7.2ms  (runs=20, avg= 359.3µs)
│       └── ]8;id=888639;file://<string>:20\Mask background]8;;\                 11%      58%    ↳ 75%  287.9ms  (runs=20, avg=  14.4ms)
├── ]8;id=340730;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:462\Geometric Augmentation]8;;\                ↳ 23%  629.7ms                    (runs=20, avg=  31.5ms)
│   ├── ]8;id=871489;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:463\generate transform]8;;\                1.37%    ↳5.9%   37.3ms           (runs=20, avg=   1.9ms)
│   ├── ]8;id=885404;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/data.py:465\transform graph]8;;\                   9.33%    ↳ 40%  253.7ms           (runs=20, avg=  12.

In [ ]:
subset.get(0, augment=True, version="fvt")
%timeit subset.get(0, augment=True, version='fvt')

In [ ]:
import cProfile

dataset.preload()

In [ ]:
cProfile.run("dataset.get(20, augment=True, version='fvt')", sort="cumulative")

In [ ]:
from fundus_vessels_toolkit.utils.profiling import ProfilerWatch

ProfilerWatch.reset()
dataset.get(20, augment=True, version="fvt")
print(ProfilerWatch.get("split_branch").print())

In [ ]:
from fundus_toolkits.transform import ElasticTransform, ElasticTransformLegacy

data = dataset.get(20, augment=False, version="fvt")

elastic = ElasticTransform.random(data.img.shape[-2:], 80, 200)
elastic_leg = ElasticTransformLegacy.random(data.img.shape[-2:], 80, 200)


%timeit ElasticTransform.random(data.img.shape[-2:], 80, 200)
%timeit ElasticTransformLegacy.random(data.img.shape[-2:], 80, 200)

In [ ]:
sample = dataset.get_sample(20)

In [ ]:
%timeit sample.graphes['fvt'].transform(elastic)
%timeit sample.graphes['fvt'].transform(elastic_leg)

In [ ]:
%timeit elastic.warp(data.img.permute(1, 2, 0))
%timeit elastic_leg.warp(data.img.permute(1, 2, 0).numpy())

In [ ]:
from fundus_vessels_toolkit.utils.profiling import watch, ProfilerWatch

elastic = ElasticTransform.random(data.img.shape[-2:], 80, 200)
elastic_leg = ElasticTransformLegacy.random(data.img.shape[-2:], 80, 200)
ProfilerWatch.reset()
elastic.warp(data.img.permute(1, 2, 0))
elastic_leg.warp(data.img.permute(1, 2, 0).numpy())
sample.graphes["fvt"].transform(elastic)
sample.graphes["fvt"].transform(elastic_leg)
print(watch("warp").print())
print(watch("ElasticTransformLegacy._warp").print())
print(watch("ElasticTransform._transform inverse").print())
print(watch("ElasticTransformLegacy._transform inverse").print())